In [ ]:
import cvxpy as cp
import numpy as np

In [65]:
def read_data(filepath):
    data = dict()
    data["F"] = []
    data["P"] = []

    with open(filepath,'r') as infile:
        infile.readline()
        data["n"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["v0"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["vmin"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["vmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["tmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["dmax"] = int(infile.readline())
        infile.readline()
        
        infile.readline()
        data["mmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["et"] = float(infile.readline())
        infile.readline()

        infile.readline()
        data["me"] = float(infile.readline())
        infile.readline()

        infile.readline()
        data["tdmin"]= int(infile.readline())
        infile.readline()

        infile.readline()
        data["vtmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        data["vdmax"] = int(infile.readline())
        infile.readline()

        infile.readline()
        for _ in range(data["n"]):
            data["F"].append(float(infile.readline()))
        infile.readline()

        infile.readline()
        for _ in range(data["n"]):
            data["P"].append(float(infile.readline()))

        data["F"] = np.array(data["F"])
        data["P"] = np.array(data["P"])
        
    return data

In [ ]:
def hydro(data):
    params = read_data(data)
    N = int(params['n']) 
    
    #Decision variables
    T = cp.Variable(N, nonneg=True, name="Turbinage")
    M = cp.Variable(N, nonneg=True, name="Pompage")
    D = cp.Variable(N, nonneg=True, name="Délestage")
    V = cp.Variable(N, nonneg=True, name="Volume")
    
    #Objective function
    profits = params['P'] @ (params['et'] * T - params['me'] * M)
    objective = cp.Maximize(profits)
    
    #list of constraints
    constraints = []
    
    #Cyclic volume constraint
    constraints += [V[0] == params['v0']]
    constraints += [V[N-1] == params['v0']]
    
    for t in range(N):
        
        #Limits 
        constraints += [V[t] >= params['vmin'], V[t] <= params['vmax']]
        constraints += [T[t] <= params['tmax']]
        constraints += [M[t] <= params['mmax']]
        constraints += [D[t] <= params['dmax']]
        
        #TDlimit
        constraints += [T[t] + D[t] >= params['tdmin']]

        if t > 0:
            #volume reservoir (V0 quand t == 0)
            constraints += [V[t] == V[t-1] + params['F'][t] + M[t] - T[t] - D[t]]
            #Derivative limits
            constraints += [(T[t] - T[t-1]) <= params['vtmax']]
            constraints += [(T[t] - T[t-1]) >= -params['vtmax']]
            constraints += [(D[t] - D[t-1]) <= params['vdmax']]
            constraints += [(D[t] - D[t-1]) >= -params['vdmax']]
            
    #solver
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.HIGHS)
    
    sol = {
        "V": V.value,       
        "T": T.value,       
        "D": D.value,       
        "M": M.value,       
        "valopt": prob.value 
    }
    
    return sol

Bénéfice optimal : 11425407.19 €


In [90]:
result = hydro("BelgiumScenario1.txt")
print(f"Bénéfice optimal : {result['valopt']:.2f} €")
result

Bénéfice optimal : 11096101.23 €


{'V': array([5000000.        , 4893975.77028029, 4937951.54056058, ...,
        5180000.        , 4990000.        , 5000000.        ]),
 'T': array([400000., 200000.,  50000., ..., 400000., 280000.,  80000.]),
 'D': array([0., 0., 0., ..., 0., 0., 0.]),
 'M': array([0., 0., 0., ..., 0., 0., 0.]),
 'valopt': np.float64(11096101.227209544)}

In [91]:
result = hydro("BelgiumScenario2.txt")
print(f"Bénéfice optimal : {result['valopt']:.2f} €")
result

Bénéfice optimal : 11425407.19 €


{'V': array([5000000.        , 4743975.77028029, 4537951.54056058, ...,
        5620000.        , 5310000.        , 5000000.        ]),
 'T': array([400000., 350000., 300000., ..., 400000., 400000., 400000.]),
 'D': array([0., 0., 0., ..., 0., 0., 0.]),
 'M': array([0., 0., 0., ..., 0., 0., 0.]),
 'valopt': np.float64(11425407.193016822)}

In [ ]:
def hydro_baseline(data):#flux turbiné compense exactement le flux entrant à chaque instant.
    params = read_data(data)
    return sum([params["P"][t]*params["et"]*params["F"][t] for t in range(params["n"])])

np.float64(8119553.744095378)

In [ ]:
hydro_baseline("BelgiumScenario1.txt") #Pareil pour both scenarios

np.float64(8119553.744095378)

In [98]:
def hydro_sans_pompage(data): #Same as hydro mais sans pompage whatsoever
    params = read_data(data)
    N = int(params['n']) 
    
    #Decision variables
    T = cp.Variable(N, nonneg=True, name="Turbinage")
    D = cp.Variable(N, nonneg=True, name="Délestage")
    V = cp.Variable(N, nonneg=True, name="Volume")
    
    #Objective function
    profits = params['P'] @ (params['et'] * T)
    objective = cp.Maximize(profits)
    
    #list of constraints
    constraints = []
    
    #Cyclic volume constraint
    constraints += [V[0] == params['v0']]
    constraints += [V[N-1] == params['v0']]
    
    for t in range(N):
        
        #Limits 
        constraints += [V[t] >= params['vmin'], V[t] <= params['vmax']]
        constraints += [T[t] <= params['tmax']]
        constraints += [D[t] <= params['dmax']]
        
        #TDlimit
        constraints += [T[t] + D[t] >= params['tdmin']]

        if t > 0:
            #volume reservoir (V0 quand t == 0)
            constraints += [V[t] == V[t-1] + params['F'][t] - T[t] - D[t]]
            #Derivative limits
            constraints += [(T[t] - T[t-1]) <= params['vtmax']]
            constraints += [(T[t] - T[t-1]) >= -params['vtmax']]
            constraints += [(D[t] - D[t-1]) <= params['vdmax']]
            constraints += [(D[t] - D[t-1]) >= -params['vdmax']]
            
    #solver
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.HIGHS)
    
    sol = {
        "V": V.value,       
        "T": T.value,       
        "D": D.value,              
        "valopt": prob.value 
    }
    
    return sol

In [103]:
hydro_sans_pompage("BelgiumScenario1.txt")

{'V': array([5000000.        , 4893975.77028029, 4937951.54056058, ...,
        4920000.        , 4960000.        , 5000000.        ]),
 'T': array([400000., 200000.,  50000., ..., 250000.,  50000.,  50000.]),
 'D': array([0., 0., 0., ..., 0., 0., 0.]),
 'valopt': np.float64(9601306.915573996)}

In [104]:
hydro_sans_pompage("BelgiumScenario2.txt")

{'V': array([5000000.        , 4743975.77028029, 4537951.54056058, ...,
        5461033.80189058, 5205516.90094529, 5000000.        ]),
 'T': array([400000.        , 350000.        , 300000.        , ...,
        295516.90094529, 345516.90094529, 295516.90094529]),
 'D': array([0., 0., 0., ..., 0., 0., 0.]),
 'valopt': np.float64(9464043.273290971)}

In [129]:
def hydro_with_duals(data_file):
    params = read_data(data_file)
    N = int(params['n']) 
    
    T = cp.Variable(N, nonneg=True)
    M = cp.Variable(N, nonneg=True)
    D = cp.Variable(N, nonneg=True)
    V = cp.Variable(N, nonneg=True)
    
    profits = params['P'] @ (params['et'] * T - params['me'] * M)
    objective = cp.Maximize(profits)
    
    constraints = []
    
    #put constraintes in variables to evaluate later
    c_vmax = [V <= params['vmax']]
    c_tmax = [T <= params['tmax']]
    c_mmax = [M <= params['mmax']]
    constraints += c_vmax + c_tmax + c_mmax
    
    #Water Volume
    c_volume = [V[0] == params['v0']]
    c_vtmax = []
    c_vdmax = []
    for t in range(1, N):
        c_volume.append(V[t] == V[t-1] + params['F'][t] + M[t] - T[t] - D[t])
        c_vtmax.append(T[t] - T[t-1] <= params['vtmax'])
        c_vtmax.append(T[t] - T[t-1] >= -params['vtmax'])
        c_vdmax.append(D[t] - D[t-1] <= params['vdmax'])
        c_vdmax.append(D[t] - D[t-1] >= -params['vdmax'])
    
    constraints += c_volume
    constraints += c_vtmax
    constraints += c_vdmax
    constraints += [V[N-1] == params['v0']]
    constraints += [V >= params['vmin']]
    constraints += [D <= params['dmax']]
    constraints += [T + D >= params['tdmin']]
    
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.HIGHS)
    

    #Valeurs duals
    #a)
    val_vmax = sum(c_vmax[0].dual_value)
    val_tmax = sum(c_tmax[0].dual_value)
    val_mmax = sum(c_mmax[0].dual_value)
    val_vtmax = sum([abs(c.dual_value) for c in c_vtmax])
    val_fk = sum([c.dual_value for c in c_volume])
    val_Pk = sum((params['et'] * T.value - params['me'] * M.value))
    
    print("Impact Vmax:",val_vmax,"€/m3")
    print(val_vmax/params['vmax'])
    print("Impact Tmax:",val_tmax,"€/(m3/h)")
    print(val_tmax/params['tmax'])
    print("Impact Mmax:",val_mmax,"€/(m3/h)")
    print(val_mmax/params['mmax'])
    print("Impact VTmax",val_vtmax,"€/(m3/h2)")
    print(val_vtmax/params['vtmax'])
    print("Impact Fk:",val_fk,"€/(m3/h)")
    print(val_fk/sum(params['F']))
    print("Impact Pk:",val_Pk,"€")
    print(val_Pk/prob.value)


In [130]:
hydro_with_duals("BelgiumScenario1.txt")

Impact Vmax: 0.5650205000000003 €/m3
9.417008333333338e-08
Impact Tmax: 3.0536385000000013 €/(m3/h)
7.634096250000003e-06
Impact Mmax: 0.9906275000000003 €/(m3/h)
1.9812550000000007e-06
Impact VTmax 0.6905314999999999 €/(m3/h2)
3.452657499999999e-06
Impact Fk: 82.689577 €/(m3/h)
5.541743932410368e-07
Impact Pk: 23931.093339291045 €
0.002156711880079816


In [131]:
hydro_with_duals("BelgiumScenario2.txt")

Impact Vmax: 0.19626375801315654 €/m3
2.453296975164457e-08
Impact Tmax: 0.9516609711373228 €/(m3/h)
2.379152427843307e-06
Impact Mmax: 2.027683346130895 €/(m3/h)
4.05536669226179e-06
Impact VTmax 12.97170236000215 €/(m3/h2)
0.000259434047200043
Impact Fk: 83.53558791452438 €/(m3/h)
5.598442442941126e-07
Impact Pk: 23540.448231890616 €
0.002060359673332122


In [133]:
hydro("extremeScenario.txt") #quitupler

{'V': array([5000000.        , 4243975.77028029, 4000000.        , ...,
        5120000.        , 4960000.        , 5000000.        ]),
 'T': array([1050000.,  850000.,  650000., ...,  450000.,  250000.,   50000.]),
 'D': array([0., 0., 0., ..., 0., 0., 0.]),
 'M': array([     0.        ,      0.        , 312048.45943942, ...,
             0.        ,      0.        ,      0.        ]),
 'valopt': np.float64(11482623.21760554)}